In [ ]:
import json
import os
import re
from collections import Counter

import pandas as pd

In [ ]:
PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"

RAW_ESCONV_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "esconv",
    "source",
    "ESConv.json"
)

print("ESConv path:")
print(RAW_ESCONV_PATH)

print("\nFile exists:", os.path.exists(RAW_ESCONV_PATH))

In [ ]:
# ============================================
# Cell 4 — Load ESConv Dataset
# ============================================

with open(RAW_ESCONV_PATH, "r", encoding="utf-8") as f:
    esconv = json.load(f)

print("Dataset loaded successfully!")
print("Number of conversations:", len(esconv))

print("\nTop-level type:", type(esconv).__name__)

if len(esconv) > 0:
    print("\nFirst conversation keys:")
    print(list(esconv[0].keys()))

In [ ]:
# ============================================
# Cell 5 — Basic Dataset Statistics
# ============================================

total_conversations = len(esconv)

total_utterances = sum(
    len(conv.get("dialog", []))
    for conv in esconv
)

speaker_counts = Counter()

for conv in esconv:
    for turn in conv.get("dialog", []):
        speaker_counts[turn.get("speaker")] += 1

print("=== ESConv Basic Statistics ===")
print("Total conversations:", total_conversations)
print("Total utterances:", total_utterances)

print("\nSpeaker distribution:")
for speaker, count in speaker_counts.items():
    print(f"{speaker}: {count}")

In [ ]:
# ============================================
# Cell 6 — Emotion Distribution
# ============================================

emotion_counts = Counter(
    conv.get("emotion_type")
    for conv in esconv
)

emotion_df = pd.DataFrame(
    emotion_counts.items(),
    columns=["Emotion", "Conversations"]
).sort_values(
    "Conversations",
    ascending=False
).reset_index(drop=True)

print("=== Emotion Distribution ===")
display(emotion_df)

In [ ]:
# ============================================
# Cell 7 — Problem Distribution
# ============================================

problem_counts = Counter(
    conv.get("problem_type")
    for conv in esconv
)

problem_df = pd.DataFrame(
    problem_counts.items(),
    columns=["Problem", "Conversations"]
).sort_values(
    "Conversations",
    ascending=False
).reset_index(drop=True)

print("=== Problem Distribution ===")
display(problem_df)

In [ ]:
# ============================================
# Cell 8 — Supporter Strategy Distribution
# ============================================

strategy_counts = Counter()

for conv in esconv:
    for turn in conv.get("dialog", []):
        if turn.get("speaker") == "supporter":
            strategy = turn.get("annotation", {}).get("strategy")

            if strategy:
                strategy_counts[strategy] += 1

strategy_df = pd.DataFrame(
    strategy_counts.items(),
    columns=["Strategy", "Utterances"]
).sort_values(
    "Utterances",
    ascending=False
).reset_index(drop=True)

print("=== Supporter Strategy Distribution ===")
display(strategy_df)

print("\nTotal annotated supporter utterances:",
      strategy_df["Utterances"].sum())

In [ ]:
# ============================================
# Cell 9 — Conversation Length Analysis
# ============================================

conversation_lengths = [
    len(conv.get("dialog", []))
    for conv in esconv
]

print("=== Conversation Length ===")
print("Minimum turns:", min(conversation_lengths))
print("Maximum turns:", max(conversation_lengths))
print("Average turns:", sum(conversation_lengths) / len(conversation_lengths))
print("Median turns:", pd.Series(conversation_lengths).median())

In [ ]:
# ============================================
# Cell 10 — Dataset Quality Check
# ============================================

empty_messages = 0
starts_with_seeker = 0
consecutive_same_speaker = 0

previous_speaker_examples = []

for i, conv in enumerate(esconv):

    dialog = conv.get("dialog", [])

    # Check first speaker
    if dialog and dialog[0].get("speaker") == "seeker":
        starts_with_seeker += 1

    previous_speaker = None

    for turn in dialog:

        speaker = turn.get("speaker")
        content = turn.get("content", "")

        # Empty content
        if not content or not content.strip():
            empty_messages += 1

        # Consecutive speaker
        if previous_speaker == speaker:
            consecutive_same_speaker += 1

        previous_speaker = speaker


print("=== ESConv Quality Check ===")
print("Conversations:", len(esconv))
print("Empty messages:", empty_messages)
print("Conversations starting with seeker:", starts_with_seeker)
print("Consecutive same-speaker turns:", consecutive_same_speaker)

In [ ]:
# ============================================
# Cell 11 — Duplicate Conversation Analysis
# ============================================

import hashlib

def normalize_text(text):
    """Normalize text for duplicate detection."""
    if not text:
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


# Create a normalized representation of each conversation
conversation_hashes = {}
duplicate_groups = {}

for i, conv in enumerate(esconv):

    dialog = conv.get("dialog", [])

    conversation_text = "\n".join(
        f"{turn.get('speaker', '').lower()}: "
        f"{normalize_text(turn.get('content', ''))}"
        for turn in dialog
    )

    conversation_hash = hashlib.md5(
        conversation_text.encode("utf-8")
    ).hexdigest()

    if conversation_hash in conversation_hashes:
        original_index = conversation_hashes[conversation_hash]

        duplicate_groups.setdefault(
            conversation_hash,
            [original_index]
        ).append(i)

    else:
        conversation_hashes[conversation_hash] = i


print("=== Duplicate Conversation Analysis ===")
print("Total conversations:", len(esconv))
print("Unique conversations:", len(conversation_hashes))
print("Exact duplicate groups:", len(duplicate_groups))

if duplicate_groups:
    print("\nDuplicate groups:")
    for group in duplicate_groups.values():
        print(group)
else:
    print("\nNo exact duplicate conversations found.")

In [ ]:
# ============================================
# Cell 12 — Duplicate Utterances Within Conversations
# ============================================

duplicate_utterance_cases = []

for i, conv in enumerate(esconv):

    seen = {}

    for turn_index, turn in enumerate(conv.get("dialog", [])):

        content = normalize_text(
            turn.get("content", "")
        )

        if not content:
            continue

        if content in seen:

            duplicate_utterance_cases.append({
                "conversation_index": i,
                "first_turn": seen[content],
                "duplicate_turn": turn_index,
                "speaker": turn.get("speaker"),
                "content": turn.get("content", "").strip()
            })

        else:
            seen[content] = turn_index


duplicate_utterance_df = pd.DataFrame(
    duplicate_utterance_cases
)

print("=== Duplicate Utterances Within Conversations ===")
print(
    "Number of duplicate utterance cases:",
    len(duplicate_utterance_df)
)

if not duplicate_utterance_df.empty:
    display(duplicate_utterance_df.head(20))

In [ ]:
# ============================================
# Cell 13 — Duplicate Situation Analysis
# ============================================

situation_counts = Counter(
    normalize_text(conv.get("situation", ""))
    for conv in esconv
    if conv.get("situation")
)

duplicate_situations = {
    situation: count
    for situation, count in situation_counts.items()
    if count > 1
}

print("=== Duplicate Situations ===")
print("Number of duplicated situations:", len(duplicate_situations))

for situation, count in duplicate_situations.items():
    print(f"\n{count}x: {situation}")

In [ ]:
# ============================================
# Cell 14 — Exact Consecutive Duplicate Turns
# ============================================

consecutive_exact_duplicates = []

for i, conv in enumerate(esconv):

    dialog = conv.get("dialog", [])

    for j in range(1, len(dialog)):

        previous = dialog[j - 1]
        current = dialog[j]

        previous_speaker = previous.get("speaker")
        current_speaker = current.get("speaker")

        previous_text = normalize_text(
            previous.get("content", "")
        )

        current_text = normalize_text(
            current.get("content", "")
        )

        if (
            previous_speaker == current_speaker
            and previous_text
            and previous_text == current_text
        ):
            consecutive_exact_duplicates.append({
                "conversation_index": i,
                "turn_1": j - 1,
                "turn_2": j,
                "speaker": current_speaker,
                "content": current.get("content", "").strip()
            })


print("=== Exact Consecutive Duplicate Turns ===")
print(
    "Number of exact consecutive duplicates:",
    len(consecutive_exact_duplicates)
)

if consecutive_exact_duplicates:
    display(
        pd.DataFrame(consecutive_exact_duplicates)
    )

In [ ]:
# ============================================
# Cell 15 — Inspect Exact Consecutive Duplicates
# ============================================

duplicate_df = pd.DataFrame(consecutive_exact_duplicates)

print("=== Exact Consecutive Duplicate Summary ===")
print("Total duplicate cases:", len(duplicate_df))

if not duplicate_df.empty:

    print("\nDuplicate cases by speaker:")
    print(duplicate_df["speaker"].value_counts())

    print("\nSample duplicate cases:")
    display(
        duplicate_df.head(20)
    )

In [ ]:
# ============================================
# Cell 16 — Simulate Duplicate Removal
# ============================================

cleaned_esconv_simulation = []

removed_duplicate_count = 0

for conv in esconv:

    dialog = conv.get("dialog", [])

    cleaned_dialog = []

    for turn in dialog:

        current_speaker = turn.get("speaker")
        current_text = normalize_text(
            turn.get("content", "")
        )

        # Check whether this turn is an exact
        # consecutive duplicate of the previous turn
        if cleaned_dialog:

            previous_turn = cleaned_dialog[-1]

            previous_speaker = previous_turn.get("speaker")
            previous_text = normalize_text(
                previous_turn.get("content", "")
            )

            if (
                current_speaker == previous_speaker
                and current_text
                and current_text == previous_text
            ):
                removed_duplicate_count += 1
                continue

        cleaned_dialog.append(turn)

    cleaned_conv = conv.copy()
    cleaned_conv["dialog"] = cleaned_dialog

    cleaned_esconv_simulation.append(cleaned_conv)


original_utterances = sum(
    len(conv.get("dialog", []))
    for conv in esconv
)

cleaned_utterances = sum(
    len(conv.get("dialog", []))
    for conv in cleaned_esconv_simulation
)

print("=== Duplicate Removal Simulation ===")
print("Original utterances:", original_utterances)
print("Removed duplicate turns:", removed_duplicate_count)
print("Remaining utterances:", cleaned_utterances)

print(
    "Percentage removed:",
    round(
        removed_duplicate_count / original_utterances * 100,
        3
    ),
    "%"
)

In [ ]:
# ============================================
# Cell 17 — Basic PII Screening
# ============================================

EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)"
)

URL_PATTERN = re.compile(
    r"https?://\S+|www\.\S+",
    re.IGNORECASE
)


pii_findings = []

for conv_idx, conv in enumerate(esconv):

    for turn_idx, turn in enumerate(conv.get("dialog", [])):

        text = turn.get("content", "")

        if not text:
            continue

        findings = []

        if EMAIL_PATTERN.search(text):
            findings.append("email")

        if PHONE_PATTERN.search(text):
            findings.append("phone")

        if URL_PATTERN.search(text):
            findings.append("url")

        if findings:
            pii_findings.append({
                "conversation_index": conv_idx,
                "turn_index": turn_idx,
                "speaker": turn.get("speaker"),
                "pii_types": findings,
                "content": text.strip()
            })


print("=== Basic PII Screening ===")
print("Turns containing possible PII:", len(pii_findings))

if pii_findings:
    display(pd.DataFrame(pii_findings))

In [ ]:
# ============================================
# Cell 18 — NER-Based PII Screening
# ============================================

import spacy

# Load English NER model
nlp = spacy.load("en_core_web_sm")

NER_ENTITY_TYPES = {
    "PERSON",
    "ORG",
    "GPE",
    "LOC",
    "FAC"
}

ner_findings = []

for conv_idx, conv in enumerate(esconv):

    for turn_idx, turn in enumerate(conv.get("dialog", [])):

        text = turn.get("content", "").strip()

        if not text:
            continue

        doc = nlp(text)

        entities = [
            {
                "text": ent.text,
                "label": ent.label_
            }
            for ent in doc.ents
            if ent.label_ in NER_ENTITY_TYPES
        ]

        if entities:
            ner_findings.append({
                "conversation_index": conv_idx,
                "turn_index": turn_idx,
                "speaker": turn.get("speaker"),
                "entities": entities,
                "content": text
            })


print("=== NER-Based PII Screening ===")
print("Turns containing potential named entities:", len(ner_findings))

if ner_findings:
    display(pd.DataFrame(ner_findings))

In [ ]:
# ============================================
# Cell 19 — Inspect PERSON Entities
# ============================================

person_findings = []

for item in ner_findings:

    persons = [
        entity
        for entity in item["entities"]
        if entity["label"] == "PERSON"
    ]

    if persons:
        person_findings.append({
            "conversation_index": item["conversation_index"],
            "turn_index": item["turn_index"],
            "speaker": item["speaker"],
            "persons": persons,
            "content": item["content"]
        })


print("=== PERSON Entity Screening ===")
print("Turns containing PERSON entities:", len(person_findings))

if person_findings:
    display(pd.DataFrame(person_findings))

In [ ]:
# ============================================
# Cell 20 — Most Frequent PERSON Entities
# ============================================

person_entity_counter = Counter()

for item in person_findings:
    for entity in item["persons"]:
        person_entity_counter[entity["text"].strip()] += 1

print("=== Most Frequent PERSON Entities ===")

for entity, count in person_entity_counter.most_common(30):
    print(f"{entity:<30} {count}")

In [ ]:
# ============================================
# Cell 21 — PII Redaction
# ============================================

def redact_pii(text):
    """
    Redact high-confidence contact information
    while preserving the rest of the utterance.
    """

    if not text:
        return text

    # Redact email addresses
    text = EMAIL_PATTERN.sub("[EMAIL]", text)

    # Redact URLs
    text = URL_PATTERN.sub("[URL]", text)

    # Redact phone numbers
    text = PHONE_PATTERN.sub("[PHONE]", text)

    return text


# Create a cleaned copy.
# The original `esconv` remains unchanged.

esconv_pii_cleaned = []

redacted_counts = Counter()

for conv in esconv:

    cleaned_conv = conv.copy()
    cleaned_dialog = []

    for turn in conv.get("dialog", []):

        cleaned_turn = turn.copy()

        original_text = turn.get("content", "")
        cleaned_text = redact_pii(original_text)

        # Track whether anything changed
        if cleaned_text != original_text:

            if EMAIL_PATTERN.search(original_text):
                redacted_counts["email"] += 1

            if URL_PATTERN.search(original_text):
                redacted_counts["url"] += 1

            if PHONE_PATTERN.search(original_text):
                redacted_counts["phone"] += 1

        cleaned_turn["content"] = cleaned_text

        cleaned_dialog.append(cleaned_turn)

    cleaned_conv["dialog"] = cleaned_dialog
    esconv_pii_cleaned.append(cleaned_conv)


print("=== PII Redaction Summary ===")
print("Original conversations:", len(esconv))
print("Cleaned conversations:", len(esconv_pii_cleaned))

print("\nRedacted turns by type:")
for pii_type, count in redacted_counts.items():
    print(f"{pii_type}: {count}")

print("\nTotal redacted turns:", sum(redacted_counts.values()))

In [ ]:
# ============================================
# Cell 22 — Verify PII Redaction
# ============================================

remaining_pii = []

for conv_idx, conv in enumerate(esconv_pii_cleaned):

    for turn_idx, turn in enumerate(conv.get("dialog", [])):

        text = turn.get("content", "")

        if (
            EMAIL_PATTERN.search(text)
            or PHONE_PATTERN.search(text)
            or URL_PATTERN.search(text)
        ):
            remaining_pii.append({
                "conversation_index": conv_idx,
                "turn_index": turn_idx,
                "speaker": turn.get("speaker"),
                "content": text.strip()
            })


print("=== PII Redaction Verification ===")
print("Remaining regex-detected PII turns:", len(remaining_pii))

if remaining_pii:
    display(pd.DataFrame(remaining_pii))
else:
    print("No remaining regex-detected email, phone, or URL patterns.")

In [ ]:
# ============================================
# Cell 23 — Initial Safety Keyword Screening
# ============================================

SAFETY_PATTERNS = {
    "suicide": [
        r"\bsuicid(e|al)\b",
        r"\bkill myself\b",
        r"\bend my life\b",
        r"\btake my own life\b",
        r"\bwant to die\b",
        r"\bwish i was dead\b",
        r"\bbetter off dead\b",
    ],

    "self_harm": [
        r"\bself[- ]harm\b",
        r"\bcut myself\b",
        r"\bhurt myself\b",
        r"\bself[- ]injur",
    ],

    "violence": [
        r"\bkill (him|her|them)\b",
        r"\bhurt (him|her|them)\b",
        r"\bwant to hurt\b",
        r"\bviolent\b",
        r"\bviolence\b",
    ],

    "substance": [
        r"\boverdose\b",
        r"\bdrug abuse\b",
        r"\bdrug addiction\b",
        r"\baddicted to drugs\b",
        r"\bheroin\b",
        r"\bcocaine\b",
        r"\bmethamphetamine\b",
    ],

    "medication": [
        r"\bprescription\b",
        r"\bmedication\b",
        r"\bmedicine\b",
        r"\bantidepressant\b",
        r"\bantipsychotic\b",
        r"\bprozac\b",
        r"\blexapro\b",
        r"\bzoloft\b",
    ],

    "diagnosis": [
        r"\bdiagnos(e|ed|is|ing)\b",
        r"\bi have depression\b",
        r"\bi have anxiety\b",
        r"\bdiagnosed with\b",
        r"\bmental disorder\b",
    ]
}


safety_findings = []

for conv_idx, conv in enumerate(esconv_pii_cleaned):

    for turn_idx, turn in enumerate(conv.get("dialog", [])):

        text = turn.get("content", "").strip()

        if not text:
            continue

        text_lower = text.lower()

        matched_categories = []

        for category, patterns in SAFETY_PATTERNS.items():

            for pattern in patterns:

                if re.search(pattern, text_lower):
                    matched_categories.append(category)
                    break

        if matched_categories:

            safety_findings.append({
                "conversation_index": conv_idx,
                "turn_index": turn_idx,
                "speaker": turn.get("speaker"),
                "categories": matched_categories,
                "content": text
            })


print("=== Initial Safety Screening ===")
print("Flagged turns:", len(safety_findings))

if safety_findings:
    display(pd.DataFrame(safety_findings).head(50))

In [ ]:
# ============================================
# Cell 24 — Safety Findings by Speaker
# ============================================

safety_df = pd.DataFrame(safety_findings)

print("=== Safety Findings by Speaker ===")

if not safety_df.empty:

    print("\nBy speaker:")
    print(safety_df["speaker"].value_counts())

    print("\nBy safety category:")
    
    category_counts = Counter()

    for categories in safety_df["categories"]:
        for category in categories:
            category_counts[category] += 1

    for category, count in category_counts.most_common():
        print(f"{category}: {count}")

In [ ]:
# ============================================
# Cell 25 — Safety Findings by Speaker & Category
# ============================================

if not safety_df.empty:

    rows = []

    for _, row in safety_df.iterrows():

        for category in row["categories"]:

            rows.append({
                "speaker": row["speaker"],
                "category": category
            })

    speaker_category_df = pd.DataFrame(rows)

    print("=== Safety Categories by Speaker ===")

    display(
        pd.crosstab(
            speaker_category_df["category"],
            speaker_category_df["speaker"]
        )
    )

In [ ]:
# ============================================
# Cell 26 — Extract Supporter Safety Cases
# ============================================

supporter_safety_cases = []

for item in safety_findings:

    if item["speaker"] != "supporter":
        continue

    conv_idx = item["conversation_index"]
    turn_idx = item["turn_index"]

    dialog = esconv_pii_cleaned[conv_idx]["dialog"]

    # Get surrounding context
    start = max(0, turn_idx - 2)
    end = min(len(dialog), turn_idx + 3)

    context = []

    for i in range(start, end):

        context.append({
            "turn_index": i,
            "speaker": dialog[i].get("speaker"),
            "content": dialog[i].get("content", "").strip()
        })

    supporter_safety_cases.append({
        "conversation_index": conv_idx,
        "turn_index": turn_idx,
        "categories": item["categories"],
        "context": context
    })


print("=== Supporter Safety Cases ===")
print(
    "Number of flagged supporter turns:",
    len(supporter_safety_cases)
)

In [ ]:
# Create a review table for flagged supporter responses

safety_review = []

for case in supporter_safety_cases:
    context_text = "\n".join(
        f"{turn['speaker']}: {turn['content']}"
        for turn in case["context"]
    )

    safety_review.append({
        "conversation_index": case["conversation_index"],
        "turn_index": case["turn_index"],
        "categories": ", ".join(case["categories"]),
        "context": context_text,
        "safety_label": "",
        "reason": ""
    })

safety_review_df = pd.DataFrame(safety_review)

print("=== Safety Review Table ===")
print("Number of supporter cases:", len(safety_review_df))
print("Columns:", list(safety_review_df.columns))

display(safety_review_df.head(10))

In [ ]:
# Manual safety labels for supporter responses
#
# KEEP   = safe to include in fine-tuning
# REVIEW = ambiguous; needs closer inspection
# REMOVE = unsafe behavior that we do not want the model to learn

VALID_LABELS = {"KEEP", "REVIEW", "REMOVE"}

def show_safety_case(case_number):
    row = safety_review_df.iloc[case_number]

    print("=" * 100)
    print(f"Case: {case_number + 1}")
    print(f"Conversation: {row['conversation_index']}")
    print(f"Turn: {row['turn_index']}")
    print(f"Category: {row['categories']}")
    print("=" * 100)
    print(row["context"])
    print()

In [ ]:
# Manual safety labels for supporter responses
#
# KEEP   = safe to include in fine-tuning
# REVIEW = ambiguous; needs closer inspection
# REMOVE = unsafe behavior that we do not want the model to learn

VALID_LABELS = {"KEEP", "REVIEW", "REMOVE"}

def show_safety_case(case_number):
    row = safety_review_df.iloc[case_number]

    print("=" * 100)
    print(f"Case: {case_number + 1}")
    print(f"Conversation: {row['conversation_index']}")
    print(f"Turn: {row['turn_index']}")
    print(f"Category: {row['categories']}")
    print("=" * 100)
    print(row["context"])
    print()

In [ ]:
show_safety_case(0)

In [ ]:
# ============================================================
# Create complete 90-case safety review file
# ============================================================

review_export = safety_review_df.copy()

# Add empty fields for manual/researcher classification
review_export["safety_label"] = review_export["safety_label"].replace("", pd.NA)
review_export["reason"] = review_export["reason"].replace("", pd.NA)

# Add a simple numeric case ID
review_export.insert(
    0,
    "case_id",
    range(1, len(review_export) + 1)
)

# Save CSV
REVIEW_CSV_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_safety_review.csv"
)

review_export.to_csv(
    REVIEW_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Safety review file created:")
print(REVIEW_CSV_PATH)
print()
print("Total cases:", len(review_export))
print()
print(review_export[
    [
        "case_id",
        "conversation_index",
        "turn_index",
        "categories",
        "safety_label",
        "reason"
    ]
].head(10))

In [ ]:
import os
import pandas as pd

REVIEW_PATH = "<PROJECT_ROOT>/mental_health_ft"

print("Exists:", os.path.exists(REVIEW_PATH))

review_df = pd.read_csv(REVIEW_PATH)

print("Shape:", review_df.shape)
print("Columns:")
print(review_df.columns.tolist())

display(review_df.head())

In [ ]:
import pandas as pd
import os

PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"

FINAL_DECISIONS_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_review_final_decisions.csv"
)

ORIGINAL_REVIEW_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_safety_review.csv"
)

final_decisions = pd.read_csv(FINAL_DECISIONS_PATH)
original_review = pd.read_csv(ORIGINAL_REVIEW_PATH)

print("Final decisions:")
display(final_decisions)

print("\nOriginal review columns:")
print(original_review.columns.tolist())

In [ ]:
import pandas as pd
import os

PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"

FINAL_DECISIONS_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_review_final_decisions.csv"
)

ORIGINAL_REVIEW_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_safety_review.csv"
)

final_decisions = pd.read_csv(FINAL_DECISIONS_PATH)
original_review = pd.read_csv(ORIGINAL_REVIEW_PATH)

review_final = original_review.merge(
    final_decisions[["case_id", "final_label", "reason"]],
    on="case_id",
    how="inner",
    suffixes=("_original", "_final")
)

print("Final reviewed cases:", len(review_final))

display(
    review_final[
        [
            "case_id",
            "conversation_index",
            "turn_index",
            "final_label",
            "reason_final"
        ]
    ]
)

In [ ]:
import pandas as pd
import os

PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"

FINAL_REVIEW_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_safety_review_final.csv"
)

print("File exists:", os.path.exists(FINAL_REVIEW_PATH))

review_df = pd.read_csv(FINAL_REVIEW_PATH)

print("Shape:", review_df.shape)
print("\nColumns:")
print(review_df.columns.tolist())

display(review_df.head())

In [ ]:
import json
import os
import pandas as pd

PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"

FINAL_REVIEW_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "esconv_safety_review_final.csv"
)

# Load final safety decisions
review_df = pd.read_csv(FINAL_REVIEW_PATH)

# Normalize labels
review_df["safety_label"] = (
    review_df["safety_label"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Total reviewed cases:", len(review_df))
print("\nSafety decisions:")
print(review_df["safety_label"].value_counts())

# Check required columns
required_columns = [
    "case_id",
    "conversation_index",
    "turn_index",
    "safety_label"
]

missing = [
    col for col in required_columns
    if col not in review_df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\n✓ All required columns are present.")

In [ ]:
remove_df = review_df[
    review_df["safety_label"] == "REMOVE"
].copy()

keep_df = review_df[
    review_df["safety_label"] == "KEEP"
].copy()

print("KEEP cases:", len(keep_df))
print("REMOVE cases:", len(remove_df))
print("Total:", len(keep_df) + len(remove_df))

In [ ]:
print("=== Turns marked REMOVE ===")

display(
    remove_df[
        [
            "case_id",
            "conversation_index",
            "turn_index",
            "categories",
            "reason"
        ]
    ].sort_values(
        ["conversation_index", "turn_index"]
    )
)

In [ ]:
remove_locations = set(
    zip(
        remove_df["conversation_index"].astype(int),
        remove_df["turn_index"].astype(int)
    )
)

print("Unique turns to remove:", len(remove_locations))

In [ ]:
# Verify that every REMOVE case points to a supporter turn

remove_locations = set(
    zip(
        remove_df["conversation_index"].astype(int),
        remove_df["turn_index"].astype(int)
    )
)

verification = []

for conv_idx, turn_idx in sorted(remove_locations):
    dialog = esconv_pii_cleaned[conv_idx]["dialog"]

    if turn_idx >= len(dialog):
        verification.append({
            "conversation_index": conv_idx,
            "turn_index": turn_idx,
            "speaker": "INVALID INDEX",
            "content": "",
            "status": "ERROR"
        })
        continue

    turn = dialog[turn_idx]

    verification.append({
        "conversation_index": conv_idx,
        "turn_index": turn_idx,
        "speaker": turn.get("speaker"),
        "content": turn.get("content", ""),
        "status": (
            "OK"
            if turn.get("speaker") == "supporter"
            else "NOT SUPPORTER"
        )
    })

verification_df = pd.DataFrame(verification)

display(verification_df)

In [ ]:
# Create the final safety-cleaned ESConv
# Original ESConv.json remains untouched.

esconv_safety_cleaned = []

removed_turns = []

for conv_idx, conv in enumerate(esconv_pii_cleaned):

    cleaned_conv = conv.copy()
    cleaned_dialog = []

    for turn_idx, turn in enumerate(conv.get("dialog", [])):

        if (conv_idx, turn_idx) in remove_locations:

            removed_turns.append({
                "conversation_index": conv_idx,
                "turn_index": turn_idx,
                "speaker": turn.get("speaker"),
                "content": turn.get("content", "")
            })

            continue

        cleaned_dialog.append(turn)

    cleaned_conv["dialog"] = cleaned_dialog
    esconv_safety_cleaned.append(cleaned_conv)

print("=== Safety Cleaning Summary ===")
print("Original conversations:", len(esconv_pii_cleaned))
print("Final conversations:", len(esconv_safety_cleaned))
print("Removed supporter turns:", len(removed_turns))

In [ ]:
removed_df = pd.DataFrame(removed_turns)

print("Removed turns:", len(removed_df))

display(removed_df)

In [ ]:
remaining_removed = []

for conv_idx, conv in enumerate(esconv_safety_cleaned):
    for turn_idx, turn in enumerate(conv.get("dialog", [])):
        if (conv_idx, turn_idx) in remove_locations:
            remaining_removed.append(
                (conv_idx, turn_idx)
            )

print("Removed turns still present:", len(remaining_removed))

In [ ]:
original_turn_count = sum(
    len(conv.get("dialog", []))
    for conv in esconv_pii_cleaned
)

cleaned_turn_count = sum(
    len(conv.get("dialog", []))
    for conv in esconv_safety_cleaned
)

print("Original total turns:", original_turn_count)
print("Cleaned total turns:", cleaned_turn_count)
print("Difference:", original_turn_count - cleaned_turn_count)
print("Expected removals:", len(remove_locations))

In [ ]:
print("=== Removed supporter responses ===")

for item in removed_turns:
    print(
        f"\nConversation {item['conversation_index']}, "
        f"original turn {item['turn_index']}"
    )
    print("Speaker:", item["speaker"])
    print("Content:", item["content"])

In [ ]:
assert len(esconv_safety_cleaned) == len(esconv_pii_cleaned)

assert all(
    item["speaker"] == "supporter"
    for item in removed_turns
)

assert (
    original_turn_count - cleaned_turn_count
    == len(remove_locations)
)

print("✓ 1300 conversations preserved")
print("✓ 18 unsafe supporter turns removed")
print("✓ No conversations were deleted")
print("✓ Removal count matches the safety review")

In [ ]:
PII_CLEANED_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "ESConv_pii_cleaned.json"
)

with open(PII_CLEANED_PATH, "w", encoding="utf-8") as f:
    json.dump(
        esconv_pii_cleaned,
        f,
        ensure_ascii=False,
        indent=2
    )

print("PII-cleaned dataset saved:")
print(PII_CLEANED_PATH)

In [ ]:
import json
import os
from collections import Counter

FINAL_ESCONV_PATH = (
    "<PROJECT_ROOT>/mental_health_ft"
    "data/processed/esconv_safety_cleaned.json"
)

with open(FINAL_ESCONV_PATH, "r", encoding="utf-8") as f:
    final_esconv = json.load(f)

print("=== Final ESConv Quality Check ===")
print("Conversations:", len(final_esconv))

total_turns = 0
speaker_counts = Counter()
empty_turns = 0
short_conversations = 0

for conv in final_esconv:
    dialog = conv.get("dialog", [])

    if len(dialog) < 2:
        short_conversations += 1

    for turn in dialog:
        total_turns += 1

        speaker = turn.get("speaker")
        speaker_counts[speaker] += 1

        if not turn.get("content", "").strip():
            empty_turns += 1

print("Total turns:", total_turns)
print("Speaker counts:", dict(speaker_counts))
print("Empty turns:", empty_turns)
print("Conversations with <2 turns:", short_conversations)

In [ ]:
import os

PROCESSED_DIR = "<PROJECT_ROOT>/mental_health_ft"

print("Files in processed:")
for name in sorted(os.listdir(PROCESSED_DIR)):
    print(name)

In [ ]:
import os
import json
import pandas as pd

PROJECT_DIR = "<PROJECT_ROOT>/mental_health_ft"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

PII_PATH = os.path.join(
    PROCESSED_DIR,
    "ESConv_pii_cleaned.json"
)

REVIEW_PATH = os.path.join(
    PROCESSED_DIR,
    "esconv_safety_review_final.csv"
)

FINAL_PATH = os.path.join(
    PROCESSED_DIR,
    "ESConv_safety_cleaned.json"
)

# Load PII-cleaned ESConv
with open(PII_PATH, "r", encoding="utf-8") as f:
    esconv_pii_cleaned = json.load(f)

# Load final safety review
review_df = pd.read_csv(REVIEW_PATH)

# Get exact locations of turns labeled REMOVE
remove_df = review_df[
    review_df["safety_label"].str.upper() == "REMOVE"
].copy()

remove_locations = set(
    zip(
        remove_df["conversation_index"].astype(int),
        remove_df["turn_index"].astype(int)
    )
)

print("Unsafe turns to remove:", len(remove_locations))

# Remove only the reviewed supporter turns
esconv_safety_cleaned = []

for conv_idx, conversation in enumerate(esconv_pii_cleaned):

    new_conversation = conversation.copy()
    new_dialog = []

    for turn_idx, turn in enumerate(conversation["dialog"]):

        if (conv_idx, turn_idx) in remove_locations:
            continue

        new_dialog.append(turn)

    new_conversation["dialog"] = new_dialog
    esconv_safety_cleaned.append(new_conversation)

# Save
with open(FINAL_PATH, "w", encoding="utf-8") as f:
    json.dump(
        esconv_safety_cleaned,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(FINAL_PATH)

In [ ]:
# Verify conversations
assert len(esconv_safety_cleaned) == len(esconv_pii_cleaned)

# Count turns before and after
original_turns = sum(
    len(conv["dialog"])
    for conv in esconv_pii_cleaned
)

cleaned_turns = sum(
    len(conv["dialog"])
    for conv in esconv_safety_cleaned
)

removed_turns = original_turns - cleaned_turns

print("Conversations:", len(esconv_safety_cleaned))
print("Turns before:", original_turns)
print("Turns after:", cleaned_turns)
print("Turns removed:", removed_turns)

assert removed_turns == len(remove_locations)

# Check no empty conversations
empty_conversations = [
    i for i, conv in enumerate(esconv_safety_cleaned)
    if len(conv["dialog"]) == 0
]

print("Empty conversations:", len(empty_conversations))

assert len(empty_conversations) == 0

# Check file exists
assert os.path.exists(FINAL_PATH)

print("\n✓ Verification passed")
print("✓ 1300 conversations preserved")
print(f"✓ {removed_turns} reviewed unsafe turns removed")
print("✓ No conversations deleted")
print("✓ Final file exists")

In [ ]:
import json
import os
import re
from collections import Counter

FINAL_PATH = "<PROJECT_ROOT>/mental_health_ft"

with open(FINAL_PATH, "r", encoding="utf-8") as f:
    esconv_final = json.load(f)

print("=== DATASET OVERVIEW ===")
print("Conversations:", len(esconv_final))

# --------------------------------------------------
# 1. Basic structure
# --------------------------------------------------

invalid_conversations = []
empty_turns = []
invalid_speakers = []
missing_content = []

valid_speakers = {"seeker", "supporter"}

for conv_idx, conv in enumerate(esconv_final):

    if "dialog" not in conv or not isinstance(conv["dialog"], list):
        invalid_conversations.append(conv_idx)
        continue

    for turn_idx, turn in enumerate(conv["dialog"]):

        if not isinstance(turn, dict):
            empty_turns.append((conv_idx, turn_idx))
            continue

        speaker = turn.get("speaker")
        content = turn.get("content")

        if speaker not in valid_speakers:
            invalid_speakers.append(
                (conv_idx, turn_idx, speaker)
            )

        if content is None or not str(content).strip():
            missing_content.append(
                (conv_idx, turn_idx)
            )

print("\n=== STRUCTURE ===")
print("Invalid conversations:", len(invalid_conversations))
print("Invalid speaker values:", len(invalid_speakers))
print("Missing/empty content:", len(missing_content))

# --------------------------------------------------
# 2. Speaker distribution
# --------------------------------------------------

speaker_counts = Counter()

for conv in esconv_final:
    for turn in conv["dialog"]:
        speaker_counts[turn["speaker"]] += 1

print("\n=== SPEAKER DISTRIBUTION ===")
print("Seeker turns:", speaker_counts["seeker"])
print("Supporter turns:", speaker_counts["supporter"])

# --------------------------------------------------
# 3. Conversation lengths
# --------------------------------------------------

conversation_lengths = [
    len(conv["dialog"])
    for conv in esconv_final
]

print("\n=== CONVERSATION LENGTH ===")
print("Minimum:", min(conversation_lengths))
print("Maximum:", max(conversation_lengths))
print("Average:", round(
    sum(conversation_lengths) / len(conversation_lengths), 2
))

# --------------------------------------------------
# 4. Exact consecutive duplicates
# --------------------------------------------------

consecutive_duplicates = []

for conv_idx, conv in enumerate(esconv_final):

    dialog = conv["dialog"]

    for i in range(1, len(dialog)):

        previous = dialog[i - 1]
        current = dialog[i]

        prev_text = previous.get("content", "").strip()
        curr_text = current.get("content", "").strip()

        if (
            previous.get("speaker") == current.get("speaker")
            and prev_text
            and prev_text == curr_text
        ):
            consecutive_duplicates.append(
                (conv_idx, i - 1, i)
            )

print("\n=== CONSECUTIVE DUPLICATES ===")
print(
    "Exact consecutive duplicate turns:",
    len(consecutive_duplicates)
)

# --------------------------------------------------
# 5. Regex PII check
# --------------------------------------------------

EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)"
)

URL_PATTERN = re.compile(
    r"https?://\S+|www\.\S+",
    re.IGNORECASE
)

pii_findings = []

for conv_idx, conv in enumerate(esconv_final):

    for turn_idx, turn in enumerate(conv["dialog"]):

        text = turn.get("content", "")

        if (
            EMAIL_PATTERN.search(text)
            or PHONE_PATTERN.search(text)
            or URL_PATTERN.search(text)
        ):
            pii_findings.append(
                (conv_idx, turn_idx)
            )

print("\n=== PII CHECK ===")
print(
    "Remaining regex-detected PII turns:",
    len(pii_findings)
)

# --------------------------------------------------
# 6. Final verdict
# --------------------------------------------------

print("\n=== FINAL QUALITY CHECK ===")

if (
    len(invalid_conversations) == 0
    and len(invalid_speakers) == 0
    and len(missing_content) == 0
    and len(pii_findings) == 0
):
    print("✓ Structural quality checks passed")
else:
    print("⚠ Some quality issues require review")

print("\nDataset is ready for the next preprocessing stage.")

In [ ]:
# Basic verification

assert len(esconv_final) == 1300

turns_before = sum(
    len(c["dialog"])
    for c in esconv_safety_cleaned
)

turns_after = sum(
    len(c["dialog"])
    for c in esconv_final
)

assert turns_before == 38347
assert turns_after == 38248
assert turns_before - turns_after == 99

# Check for remaining exact consecutive duplicates

remaining_duplicates = []

for conv_idx, conversation in enumerate(esconv_final):

    dialog = conversation["dialog"]

    for i in range(1, len(dialog)):

        prev = dialog[i - 1]
        curr = dialog[i]

        if (
            prev.get("speaker") == curr.get("speaker")
            and prev.get("content", "").strip()
            and prev.get("content", "").strip()
                == curr.get("content", "").strip()
        ):
            remaining_duplicates.append(
                (conv_idx, i - 1, i)
            )

print("Conversations:", len(esconv_final))
print("Final turns:", turns_after)
print("Duplicates removed:", turns_before - turns_after)
print("Remaining consecutive duplicates:", len(remaining_duplicates))

assert len(remaining_duplicates) == 0
assert os.path.exists(OUTPUT_PATH)

print("\n✓ FINAL ESConv CLEANING PASSED")
print("✓ 1,300 conversations preserved")
print("✓ 99 exact duplicates removed")
print("✓ 38,248 turns remaining")
print("✓ No consecutive duplicates remain")

In [ ]:
print("Safety-cleaned dataset:")
print("Conversations:", len(esconv_safety_cleaned))
print("Turns:", sum(len(c["dialog"]) for c in esconv_safety_cleaned))

print("\nCurrent esconv_final:")
print("Conversations:", len(esconv_final))
print("Turns:", sum(len(c["dialog"]) for c in esconv_final))

print("\nExpected if 99 were removed:")
print(38347 - 99)

In [ ]:
import json
import os

INPUT_PATH = "<PROJECT_ROOT>/mental_health_ft"
OUTPUT_PATH = "<PROJECT_ROOT>/mental_health_ft"

# Load the safety-cleaned dataset
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

cleaned_data = []
removed_count = 0

for conversation in data:

    new_conversation = conversation.copy()
    new_dialog = []

    for turn in conversation["dialog"]:

        if new_dialog:
            previous_turn = new_dialog[-1]

            same_speaker = (
                previous_turn["speaker"] == turn["speaker"]
            )

            same_content = (
                previous_turn["content"].strip()
                == turn["content"].strip()
            )

            if same_speaker and same_content:
                removed_count += 1
                continue

        new_dialog.append(turn)

    new_conversation["dialog"] = new_dialog
    cleaned_data.append(new_conversation)

# Save final dataset
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        cleaned_data,
        f,
        ensure_ascii=False,
        indent=2
    )

turns_before = sum(len(c["dialog"]) for c in data)
turns_after = sum(len(c["dialog"]) for c in cleaned_data)

print("=== DUPLICATE REMOVAL ===")
print("Conversations:", len(cleaned_data))
print("Turns before:", turns_before)
print("Turns after:", turns_after)
print("Duplicates removed:", removed_count)

print("\nSaved:")
print(OUTPUT_PATH)

In [ ]:
assert len(cleaned_data) == 1300
assert turns_before == 38347
assert removed_count == 99
assert turns_after == 38248
assert os.path.exists(OUTPUT_PATH)

print("✓ 1,300 conversations preserved")
print("✓ 99 exact consecutive duplicates removed")
print("✓ 38,248 turns remaining")
print("✓ Final ESConv dataset saved successfully")

In [ ]:
import json
import os
import random

INPUT_PATH = "<PROJECT_ROOT>/mental_health_ft"

SPLIT_DIR = "<PROJECT_ROOT>/mental_health_ft"
os.makedirs(SPLIT_DIR, exist_ok=True)

# Load final cleaned ESConv
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    conversations = json.load(f)

print("Total conversations:", len(conversations))

# Reproducible shuffle
SEED = 42
random.seed(SEED)

indices = list(range(len(conversations)))
random.shuffle(indices)

# 70 / 15 / 15
n = len(indices)

n_train = int(0.70 * n)
n_validation = int(0.15 * n)

train_indices = indices[:n_train]
validation_indices = indices[n_train:n_train + n_validation]
test_indices = indices[n_train + n_validation:]

train_data = [conversations[i] for i in train_indices]
validation_data = [conversations[i] for i in validation_indices]
test_data = [conversations[i] for i in test_indices]

# Save
train_path = os.path.join(SPLIT_DIR, "ESConv_train.json")
validation_path = os.path.join(SPLIT_DIR, "ESConv_validation.json")
test_path = os.path.join(SPLIT_DIR, "ESConv_test.json")

for path, data in [
    (train_path, train_data),
    (validation_path, validation_data),
    (test_path, test_data)
]:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

print("\n=== SPLIT RESULTS ===")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nSaved files:")
print(train_path)
print(validation_path)
print(test_path)

In [ ]:
# Verify split sizes
assert len(train_data) == 910
assert len(validation_data) == 195
assert len(test_data) == 195

# Verify all conversations are unique across splits
train_ids = {id(c) for c in train_data}
validation_ids = {id(c) for c in validation_data}
test_ids = {id(c) for c in test_data}

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

# Verify total
assert (
    len(train_data)
    + len(validation_data)
    + len(test_data)
    == 1300
)

print("✓ Train: 910 conversations")
print("✓ Validation: 195 conversations")
print("✓ Test: 195 conversations")
print("✓ Total: 1,300 conversations")
print("✓ No conversation overlap")
print("✓ Random seed: 42")
print("✓ Split completed successfully")

In [ ]:
import json
import os

SPLIT_DIR = "<PROJECT_ROOT>/mental_health_ft"
CHAT_DIR = "<PROJECT_ROOT>/mental_health_ft"

os.makedirs(CHAT_DIR, exist_ok=True)


def convert_to_chat_format(conversations):
    converted = []

    for conversation in conversations:
        messages = []

        for turn in conversation["dialog"]:
            speaker = turn["speaker"]
            content = turn["content"].strip()

            if not content:
                continue

            if speaker == "seeker":
                role = "user"
            elif speaker == "supporter":
                role = "assistant"
            else:
                continue

            messages.append({
                "role": role,
                "content": content
            })

        if messages:
            converted.append({
                "messages": messages
            })

    return converted


# Load splits
with open(os.path.join(SPLIT_DIR, "ESConv_train.json"), "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open(os.path.join(SPLIT_DIR, "ESConv_validation.json"), "r", encoding="utf-8") as f:
    validation_raw = json.load(f)

with open(os.path.join(SPLIT_DIR, "ESConv_test.json"), "r", encoding="utf-8") as f:
    test_raw = json.load(f)


# Convert
train_chat = convert_to_chat_format(train_raw)
validation_chat = convert_to_chat_format(validation_raw)
test_chat = convert_to_chat_format(test_raw)


# Save
paths = {
    "train": os.path.join(CHAT_DIR, "ESConv_train_chat.json"),
    "validation": os.path.join(CHAT_DIR, "ESConv_validation_chat.json"),
    "test": os.path.join(CHAT_DIR, "ESConv_test_chat.json")
}

for name, data in [
    ("train", train_chat),
    ("validation", validation_chat),
    ("test", test_chat)
]:
    with open(paths[name], "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

print("=== CHAT FORMAT CREATED ===")
print("Train:", len(train_chat))
print("Validation:", len(validation_chat))
print("Test:", len(test_chat))

print("\nSaved:")
for path in paths.values():
    print(path)

In [ ]:
print(json.dumps(
    train_chat[0],
    ensure_ascii=False,
    indent=2
))

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Chat template available:", tokenizer.chat_template is not None)

In [ ]:
%pip install -U transformers

In [ ]:
import transformers

print("Transformers version:", transformers.__version__)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Chat template available:", tokenizer.chat_template is not None)

In [ ]:
example_messages = train_chat[0]["messages"]

formatted_text = tokenizer.apply_chat_template(
    example_messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text[:3000])

In [ ]:
from collections import Counter

first_roles = Counter()
consecutive_role_pairs = Counter()

for example in train_chat:
    messages = example["messages"]

    if messages:
        first_roles[messages[0]["role"]] += 1

    for prev, curr in zip(messages, messages[1:]):
        if prev["role"] == curr["role"]:
            consecutive_role_pairs[(prev["role"], curr["role"])] += 1

print("First-role distribution:")
print(first_roles)

print("\nConsecutive same-role pairs:")
print(consecutive_role_pairs)

In [ ]:
# Find the first training conversation that starts with user
user_first_example = next(
    ex for ex in train_chat
    if ex["messages"][0]["role"] == "user"
)

formatted_text = tokenizer.apply_chat_template(
    user_first_example["messages"],
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text[:3000])

In [ ]:
# Show a few assistant-first conversations
count = 0

for i, example in enumerate(train_chat):
    messages = example["messages"]

    if messages[0]["role"] == "assistant":
        print(f"\n--- Conversation {i} ---")
        for msg in messages[:5]:
            print(f"{msg['role']}: {msg['content']}")
        
        count += 1
        if count == 5:
            break

In [ ]:
def merge_consecutive_roles(messages):
    """
    Merge consecutive messages with the same role.
    Original message order and content are preserved.
    """
    merged = []

    for message in messages:
        role = message["role"]
        content = message["content"].strip()

        if merged and merged[-1]["role"] == role:
            merged[-1]["content"] += "\n" + content
        else:
            merged.append({
                "role": role,
                "content": content
            })

    return merged

In [ ]:
test_messages = train_chat[0]["messages"]

merged_messages = merge_consecutive_roles(test_messages)

print("Original messages:", len(test_messages))
print("After merging:", len(merged_messages))

for msg in merged_messages[:6]:
    print(f"\n[{msg['role']}]")
    print(msg["content"])

In [ ]:
# Check whether consecutive roles were successfully merged

remaining_consecutive = 0

for example in train_chat:
    merged = merge_consecutive_roles(example["messages"])

    for prev, curr in zip(merged, merged[1:]):
        if prev["role"] == curr["role"]:
            remaining_consecutive += 1

print("Remaining consecutive same-role pairs:", remaining_consecutive)

In [ ]:
def prepare_for_mistral_template(messages):
    messages = merge_consecutive_roles(messages)

    # Mistral requires the conversation to start with user
    if messages and messages[0]["role"] == "assistant":
        messages = [
            {"role": "user", "content": ""},
            *messages
        ]

    return messages


# Test on an assistant-first conversation
test_messages = train_chat[0]["messages"]

prepared_messages = prepare_for_mistral_template(test_messages)

for msg in prepared_messages[:6]:
    print(f"\n[{msg['role']}]")
    print(repr(msg["content"]))

In [ ]:
def format_for_mistral(messages):
    """
    Prepare ESConv messages for Mistral training.

    - Merges consecutive messages from the same role.
    - Preserves all original content and order.
    - Does NOT add synthetic user/assistant messages.
    """
    return merge_consecutive_roles(messages)


# Test on an assistant-first conversation
assistant_first = next(
    ex for ex in train_chat
    if ex["messages"][0]["role"] == "assistant"
)

formatted = format_for_mistral(assistant_first["messages"])

print("Number of messages:", len(formatted))
print("First role:", formatted[0]["role"])

for msg in formatted[:6]:
    print(f"\n[{msg['role']}]")
    print(msg["content"])

In [ ]:
formatted_train = [
    {
        "messages": format_for_mistral(example["messages"])
    }
    for example in train_chat
]

print("Training conversations:", len(formatted_train))

# Verify no consecutive same-role messages remain
remaining = 0

for example in formatted_train:
    messages = example["messages"]

    for prev, curr in zip(messages, messages[1:]):
        if prev["role"] == curr["role"]:
            remaining += 1

print("Remaining consecutive same-role pairs:", remaining)

# Verify content was not lost
original_chars = sum(
    len(msg["content"])
    for example in train_chat
    for msg in example["messages"]
)

formatted_chars = sum(
    len(msg["content"])
    for example in formatted_train
    for msg in example["messages"]
)

print("Original characters:", original_chars)
print("Formatted characters:", formatted_chars)

In [ ]:
print("train_chat:", len(train_chat))

print("formatted_train:", len(formatted_train))

print("Expected train conversations:", 910)

print("Difference:", 910 - len(train_chat))

In [ ]:
import json
from pathlib import Path

split_dir = Path("<PROJECT_ROOT>/mental_health_ft")

for name in ["ESConv_train.json", "ESConv_validation.json", "ESConv_test.json"]:
    path = split_dir / name

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(name, "→", len(data))

In [ ]:
import json
import random
from pathlib import Path

# Paths
base_dir = Path("<PROJECT_ROOT>/mental_health_ft")
final_file = base_dir / "data/processed/ESConv_final_cleaned.json"
split_dir = base_dir / "data/processed/splits"

# Load final cleaned dataset
with open(final_file, "r", encoding="utf-8") as f:
    conversations = json.load(f)

print("Total conversations:", len(conversations))

# Reproducible conversation-level split
indices = list(range(len(conversations)))

random.seed(42)
random.shuffle(indices)

train_indices = indices[:910]
validation_indices = indices[910:1105]
test_indices = indices[1105:1300]

print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))
print("Total:", len(train_indices) + len(validation_indices) + len(test_indices))

# Create splits
train_data = [conversations[i] for i in train_indices]
validation_data = [conversations[i] for i in validation_indices]
test_data = [conversations[i] for i in test_indices]

# Save
split_dir.mkdir(parents=True, exist_ok=True)

with open(split_dir / "ESConv_train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(split_dir / "ESConv_validation.json", "w", encoding="utf-8") as f:
    json.dump(validation_data, f, ensure_ascii=False, indent=2)

with open(split_dir / "ESConv_test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

print("\nSplit files recreated successfully.")

In [ ]:
import json
from pathlib import Path

base_dir = Path("<PROJECT_ROOT>/mental_health_ft")
split_dir = base_dir / "data/processed/splits"
chat_dir = base_dir / "data/processed/chat"

chat_dir.mkdir(parents=True, exist_ok=True)


def convert_to_chat_format(data):
    converted = []

    for conversation in data:
        messages = []

        for turn in conversation["dialog"]:
            role = "user" if turn["speaker"] == "seeker" else "assistant"

            messages.append({
                "role": role,
                "content": turn["content"]
            })

        converted.append({
            "messages": messages
        })

    return converted


# Load corrected splits
with open(split_dir / "ESConv_train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(split_dir / "ESConv_validation.json", "r", encoding="utf-8") as f:
    validation_data = json.load(f)

with open(split_dir / "ESConv_test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)


# Convert
train_chat = convert_to_chat_format(train_data)
validation_chat = convert_to_chat_format(validation_data)
test_chat = convert_to_chat_format(test_data)


# Save
with open(chat_dir / "ESConv_train_chat.json", "w", encoding="utf-8") as f:
    json.dump(train_chat, f, ensure_ascii=False, indent=2)

with open(chat_dir / "ESConv_validation_chat.json", "w", encoding="utf-8") as f:
    json.dump(validation_chat, f, ensure_ascii=False, indent=2)

with open(chat_dir / "ESConv_test_chat.json", "w", encoding="utf-8") as f:
    json.dump(test_chat, f, ensure_ascii=False, indent=2)


print("Train:", len(train_chat))
print("Validation:", len(validation_chat))
print("Test:", len(test_chat))

In [ ]:
# Apply the formatter to the corrected training set

formatted_train = [
    {
        "messages": format_for_mistral(example["messages"])
    }
    for example in train_chat
]

print("Formatted training conversations:", len(formatted_train))

# Check consecutive same-role messages
remaining = 0

for example in formatted_train:
    messages = example["messages"]

    for prev, curr in zip(messages, messages[1:]):
        if prev["role"] == curr["role"]:
            remaining += 1

print("Remaining consecutive same-role pairs:", remaining)

In [ ]:
lengths = []

for example in formatted_train:
    text_parts = []

    for message in example["messages"]:
        role = message["role"]
        content = message["content"]

        text_parts.append(
            f"<|{role}|>\n{content}"
        )

    text = "\n".join(text_parts)

    token_ids = tokenizer(
        text,
        add_special_tokens=True
    )["input_ids"]

    lengths.append(len(token_ids))


print("Number of training conversations:", len(lengths))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", round(sum(lengths) / len(lengths), 1))

sorted_lengths = sorted(lengths)

for percentile in [50, 75, 90, 95, 99]:
    index = int(len(sorted_lengths) * percentile / 100)
    index = min(index, len(sorted_lengths) - 1)

    print(
        f"{percentile}th percentile:",
        sorted_lengths[index]
    )

In [ ]:
MAX_SEQ_LENGTH = 1536

def build_training_text(messages):
    """
    Build a simple role-marked training sequence.
    The original message order is preserved.
    """

    parts = []

    for message in messages:
        role = message["role"]
        content = message["content"].strip()

        parts.append(f"<|{role}|>\n{content}")

    return "\n".join(parts)


# Test on one formatted conversation
example = formatted_train[0]

training_text = build_training_text(example["messages"])

print(training_text[:3000])

In [ ]:
print("Chat template:")
print(tokenizer.chat_template)

print("\nSpecial tokens:")
print(tokenizer.special_tokens_map)

print("\nAll special tokens:")
print(tokenizer.all_special_tokens)

In [ ]:
def render_mistral_messages(messages):
    """
    Render ESConv messages using Mistral's native instruction format.

    Consecutive same-role messages should already be merged.
    No synthetic messages are added.
    """

    rendered = []
    
    for i, message in enumerate(messages):
        role = message["role"]
        content = message["content"].strip()

        if role == "user":
            rendered.append(f"[INST] {content} [/INST]")

        elif role == "assistant":
            rendered.append(f" {content}</s>")

        else:
            raise ValueError(f"Unexpected role: {role}")

    return "<s>" + "".join(rendered)

In [ ]:
example = formatted_train[0]

rendered_text = render_mistral_messages(
    example["messages"]
)

print(rendered_text[:4000])

In [ ]:
tokens = tokenizer(
    rendered_text,
    add_special_tokens=False
)["input_ids"]

print("Token count:", len(tokens))
print("First 50 token IDs:", tokens[:50])

In [ ]:
def tokenize_with_assistant_mask(messages, tokenizer):
    """
    Tokenize one ESConv conversation using Mistral-style formatting.

    Loss is calculated only on assistant responses.
    User messages and formatting tokens are masked with -100.
    """

    input_ids = []
    labels = []

    # BOS
    bos_ids = tokenizer(
        "<s>",
        add_special_tokens=False
    )["input_ids"]

    input_ids.extend(bos_ids)
    labels.extend([-100] * len(bos_ids))

    for message in messages:

        role = message["role"]
        content = message["content"].strip()

        if role == "user":

            text = f"[INST] {content} [/INST]"

            ids = tokenizer(
                text,
                add_special_tokens=False
            )["input_ids"]

            input_ids.extend(ids)
            labels.extend([-100] * len(ids))

        elif role == "assistant":

            text = f" {content}</s>"

            ids = tokenizer(
                text,
                add_special_tokens=False
            )["input_ids"]

            input_ids.extend(ids)

            # Assistant tokens are training targets
            labels.extend(ids)

        else:
            raise ValueError(f"Unexpected role: {role}")

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids)
    }

In [ ]:
example = formatted_train[0]

encoded = tokenize_with_assistant_mask(
    example["messages"],
    tokenizer
)

print("Input tokens :", len(encoded["input_ids"]))
print("Label tokens :", len(encoded["labels"]))

print(
    "Trainable tokens:",
    sum(label != -100 for label in encoded["labels"])
)

print(
    "Masked tokens:",
    sum(label == -100 for label in encoded["labels"])
)

In [ ]:
for i, (token_id, label) in enumerate(
    zip(encoded["input_ids"], encoded["labels"])
):

    if i >= 150:
        break

    token = tokenizer.decode([token_id])

    print(
        i,
        repr(token),
        "TRAIN" if label != -100 else "MASK"
    )

In [ ]:
MAX_SEQ_LENGTH = 1536

lengths = []
trainable_lengths = []

for example in formatted_train:
    encoded = tokenize_with_assistant_mask(
        example["messages"],
        tokenizer
    )

    lengths.append(len(encoded["input_ids"]))
    trainable_lengths.append(
        sum(label != -100 for label in encoded["labels"])
    )

print("Training conversations:", len(lengths))
print("Max tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))
print("Over 1536:", sum(x > MAX_SEQ_LENGTH for x in lengths))
print("Total trainable tokens:", sum(trainable_lengths))

In [ ]:
import numpy as np

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"{p}th percentile:", np.percentile(lengths, p))

In [ ]:
long_examples = []

for idx, example in enumerate(formatted_train):
    encoded = tokenize_with_assistant_mask(
        example["messages"],
        tokenizer
    )

    total_tokens = len(encoded["input_ids"])

    if total_tokens > MAX_SEQ_LENGTH:
        trainable_total = sum(
            label != -100
            for label in encoded["labels"]
        )

        trainable_after_limit = sum(
            label != -100
            for label in encoded["labels"][MAX_SEQ_LENGTH:]
        )

        long_examples.append({
            "index": idx,
            "total_tokens": total_tokens,
            "tokens_over_limit": total_tokens - MAX_SEQ_LENGTH,
            "trainable_tokens_total": trainable_total,
            "trainable_tokens_lost": trainable_after_limit
        })

import pandas as pd

pd.DataFrame(long_examples)

In [ ]:
MAX_SEQ_LENGTH = 1536


def token_len(text):
    return len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )


def render_message(message):
    role = message["role"]
    content = message["content"].strip()

    if role == "user":
        return f"[INST] {content} [/INST]"
    elif role == "assistant":
        return f" {content}</s>"
    else:
        raise ValueError(f"Unexpected role: {role}")


def build_response_aware_chunks(messages, max_length=1536):
    """
    Build chunks without splitting assistant responses.

    - Preserves original messages.
    - Does not add synthetic messages.
    - Keeps Mistral formatting.
    - Chunks only when the current response would exceed max_length.
    """

    chunks = []
    current_messages = []
    current_tokens = token_len("<s>")

    for message in messages:
        message_tokens = token_len(
            render_message(message)
        )

        # If adding this message exceeds the limit,
        # save the current chunk first.
        if (
            current_messages
            and current_tokens + message_tokens > max_length
        ):
            chunks.append(current_messages)

            current_messages = []
            current_tokens = token_len("<s>")

        current_messages.append(message)
        current_tokens += message_tokens

    if current_messages:
        chunks.append(current_messages)

    return chunks

In [ ]:
long_chunk_results = []

for row in long_examples:

    idx = row["index"]

    messages = formatted_train[idx]["messages"]

    chunks = build_response_aware_chunks(
        messages,
        MAX_SEQ_LENGTH
    )

    chunk_lengths = []

    for chunk in chunks:
        rendered = "<s>" + "".join(
            render_message(message)
            for message in chunk
        )

        chunk_lengths.append(
            token_len(rendered)
        )

    long_chunk_results.append({
        "index": idx,
        "original_tokens": row["total_tokens"],
        "num_chunks": len(chunks),
        "chunk_lengths": chunk_lengths,
        "max_chunk_tokens": max(chunk_lengths)
    })

pd.DataFrame(long_chunk_results)

In [ ]:
for result in long_chunk_results:
    idx = result["index"]
    chunks = build_response_aware_chunks(
        formatted_train[idx]["messages"],
        MAX_SEQ_LENGTH
    )

    print(f"\nConversation {idx}")

    for i, chunk in enumerate(chunks):
        roles = [m["role"] for m in chunk]

        print(
            f"  Chunk {i + 1}: "
            f"{len(chunk)} messages | "
            f"roles = {roles}"
        )

In [ ]:
MAX_SEQ_LENGTH = 1536


def build_response_aware_chunks(messages, max_length=1536):
    """
    Split conversations at user-turn boundaries.

    Rules:
    - Never split an assistant response.
    - The first chunk may start with assistant if the original
      conversation starts with assistant.
    - Every subsequent chunk starts with user.
    - No synthetic messages are added.
    - No original message content is modified.
    """

    chunks = []
    current = []
    current_tokens = token_len("<s>")

    for message in messages:
        message_tokens = token_len(
            render_message(message)
        )

        # If adding this message exceeds the limit,
        # finalize the current chunk.
        if current and current_tokens + message_tokens > max_length:

            chunks.append(current)

            current = []
            current_tokens = token_len("<s>")

            # We should only reach here at a message boundary.
            # The next message becomes the first message of the new chunk.

        current.append(message)
        current_tokens += message_tokens

    if current:
        chunks.append(current)

    # Validate chunk boundaries.
    for i, chunk in enumerate(chunks):
        if i > 0:
            assert chunk[0]["role"] == "user", (
                f"Chunk {i} starts with "
                f"{chunk[0]['role']} instead of user."
            )

    return chunks

In [ ]:
def build_response_aware_chunks(messages, max_length=1536):
    """
    Chunk ESConv at user-turn boundaries.

    - Never split an assistant response.
    - First chunk may start with assistant.
    - Every later chunk starts with user.
    - No synthetic content.
    """

    chunks = []
    current = []
    current_tokens = token_len("<s>")

    i = 0

    while i < len(messages):

        message = messages[i]

        # --------------------------------------------------
        # Case 1: assistant-first message
        # Only allowed at the beginning of the conversation.
        # --------------------------------------------------
        if i == 0 and message["role"] == "assistant":

            message_tokens = token_len(
                render_message(message)
            )

            current.append(message)
            current_tokens += message_tokens
            i += 1
            continue

        # --------------------------------------------------
        # Normal case: user followed by assistant(s)
        # --------------------------------------------------
        if message["role"] != "user":
            raise ValueError(
                f"Unexpected role at position {i}: "
                f"{message['role']}"
            )

        user_message = message
        user_tokens = token_len(
            render_message(user_message)
        )

        # Find the assistant response(s) belonging to this user.
        j = i + 1
        assistant_messages = []

        while j < len(messages) and messages[j]["role"] == "assistant":
            assistant_messages.append(messages[j])
            j += 1

        group_tokens = user_tokens

        for assistant_message in assistant_messages:
            group_tokens += token_len(
                render_message(assistant_message)
            )

        # --------------------------------------------------
        # If current + this complete exchange fits
        # --------------------------------------------------
        if (
            current
            and current_tokens + group_tokens <= max_length
        ):
            current.append(user_message)
            current_tokens += user_tokens

            for assistant_message in assistant_messages:
                current.append(assistant_message)
                current_tokens += token_len(
                    render_message(assistant_message)
                )

            i = j
            continue

        # --------------------------------------------------
        # If current is full, save it and start new chunk.
        # --------------------------------------------------
        if current:
            chunks.append(current)
            current = []
            current_tokens = token_len("<s>")

        # --------------------------------------------------
        # Add the complete user → assistant exchange.
        # --------------------------------------------------
        if current_tokens + group_tokens <= max_length:

            current.append(user_message)
            current_tokens += user_tokens

            for assistant_message in assistant_messages:
                current.append(assistant_message)
                current_tokens += token_len(
                    render_message(assistant_message)
                )

            i = j
            continue

        # --------------------------------------------------
        # Extremely long individual exchange.
        # Don't silently split it.
        # --------------------------------------------------
        raise ValueError(
            f"Single user/assistant exchange exceeds "
            f"{max_length} tokens: {group_tokens}"
        )

    if current:
        chunks.append(current)

    # ------------------------------------------------------
    # Final validation
    # ------------------------------------------------------

    for chunk_idx, chunk in enumerate(chunks):

        chunk_text = "<s>" + "".join(
            render_message(message)
            for message in chunk
        )

        length = token_len(chunk_text)

        assert length <= max_length, (
            f"Chunk {chunk_idx} has {length} tokens."
        )

        if chunk_idx > 0:
            assert chunk[0]["role"] == "user", (
                f"Chunk {chunk_idx} starts with "
                f"{chunk[0]['role']}."
            )

    return chunks

In [ ]:
results = []

for row in long_examples:

    idx = row["index"]

    chunks = build_response_aware_chunks(
        formatted_train[idx]["messages"],
        MAX_SEQ_LENGTH
    )

    chunk_lengths = []

    for chunk in chunks:

        rendered = "<s>" + "".join(
            render_message(message)
            for message in chunk
        )

        chunk_lengths.append(
            token_len(rendered)
        )

    results.append({
        "index": idx,
        "original_tokens": row["total_tokens"],
        "num_chunks": len(chunks),
        "chunk_lengths": chunk_lengths,
        "max_chunk_tokens": max(chunk_lengths),
        "first_roles": [
            chunk[0]["role"]
            for chunk in chunks
        ]
    })

pd.DataFrame(results)

In [ ]:
# Final validation of ESConv training chunks

MAX_SEQ_LENGTH = 1536

final_examples = []

# Build final examples:
# - Normal conversations remain one example
# - Only long conversations are split into chunks

long_indices = {row["index"] for row in long_examples}

for idx, example in enumerate(formatted_train):

    messages = example["messages"]

    if idx in long_indices:
        chunks = build_response_aware_chunks(
            messages,
            MAX_SEQ_LENGTH
        )
    else:
        chunks = [messages]

    for chunk in chunks:
        encoded = tokenize_with_assistant_mask(
            chunk,
            tokenizer
        )

        final_examples.append({
            "messages": chunk,
            "input_ids": encoded["input_ids"],
            "labels": encoded["labels"],
            "attention_mask": encoded["attention_mask"]
        })

print("Final training examples:", len(final_examples))

In [ ]:
# Comprehensive validation

errors = []

for idx, example in enumerate(final_examples):

    input_ids = example["input_ids"]
    labels = example["labels"]
    attention_mask = example["attention_mask"]
    messages = example["messages"]

    # 1. Same lengths
    if not (
        len(input_ids)
        == len(labels)
        == len(attention_mask)
    ):
        errors.append(
            (idx, "length_mismatch")
        )

    # 2. Maximum sequence length
    if len(input_ids) > MAX_SEQ_LENGTH:
        errors.append(
            (idx, f"too_long:{len(input_ids)}")
        )

    # 3. Must contain assistant training tokens
    trainable = sum(
        label != -100
        for label in labels
    )

    if trainable == 0:
        errors.append(
            (idx, "no_trainable_tokens")
        )

    # 4. No empty messages
    for message in messages:
        if not message["content"].strip():
            errors.append(
                (idx, "empty_message")
            )

    # 5. Valid roles
    for message in messages:
        if message["role"] not in {"user", "assistant"}:
            errors.append(
                (idx, "invalid_role")
            )

print("Validation errors:", len(errors))

if errors:
    print(errors[:20])
else:
    print("✓ All validation checks passed")

In [ ]:
# Verify that chunking preserved all original messages

def flatten_messages(chunks):
    flattened = []

    for chunk in chunks:
        flattened.extend(chunk)

    return flattened


message_mismatch = []

for idx, example in enumerate(formatted_train):

    original_messages = example["messages"]

    chunks = build_response_aware_chunks(
        original_messages,
        MAX_SEQ_LENGTH
    )

    reconstructed = flatten_messages(chunks)

    if reconstructed != original_messages:
        message_mismatch.append(idx)

print("Message reconstruction mismatches:", len(message_mismatch))

if message_mismatch:
    print("First mismatches:", message_mismatch[:10])
else:
    print("✓ All original messages preserved exactly")

In [ ]:
import json
from pathlib import Path

FINAL_DIR = Path(
    "<PROJECT_ROOT>/mental_health_ft"
)

FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_path = FINAL_DIR / "ESConv_train_qlora.json"

with open(final_path, "w", encoding="utf-8") as f:
    json.dump(
        final_examples,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved: {final_path}")
print(f"Examples: {len(final_examples)}")